# ML Assignment 2 - Classification Models
**Student ID:** 2025AC05601  
**Dataset:** Breast Cancer Wisconsin (Diagnostic) - UCI ML Repository

This notebook implements 5 classification models and evaluates them using Accuracy, AUC, Precision, Recall, F1 Score, and MCC.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)

RANDOM_STATE = 42
print("Libraries imported successfully!")

## Step 1: Load and Explore Dataset

In [ ]:
# Load Breast Cancer Wisconsin dataset (from UCI via sklearn)
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print(f"Number of samples: {df.shape[0]}")
print(f"Number of features: {df.shape[1] - 1}")
print(f"\nTarget distribution:")
print(df['target'].value_counts())
print(f"\n0 = Malignant, 1 = Benign")
df.head()

## Step 2: Train-Test Split

In [ ]:
feature_names = list(data.feature_names)

train_df, test_df = train_test_split(
    df, test_size=0.25, random_state=RANDOM_STATE, stratify=df['target']
)

X_train = train_df[feature_names]
y_train = train_df['target']
X_test = test_df[feature_names]
y_test = test_df['target']

# Save test data for Streamlit app
test_df.to_csv('../test_data.csv', index=False)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## Step 3: Train All 5 Classification Models

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'kNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    'Naive Bayes': Pipeline([
        ('scaler', StandardScaler()),
        ('model', GaussianNB())
    ]),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Trained: {name}")

## Step 4: Evaluate Models (6 Metrics)

In [ ]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'AUC': round(roc_auc_score(y_test, y_prob), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1': round(f1_score(y_test, y_pred), 4),
        'MCC': round(matthews_corrcoef(y_test, y_pred), 4)
    })

results_df = pd.DataFrame(results)
results_df

## Step 5: Confusion Matrix (Best Model - Logistic Regression)

In [ ]:
best_model = models['Logistic Regression']
y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Logistic Regression')
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))

## Step 6: Save Models

In [ ]:
for name, model in models.items():
    filename = name.lower().replace(' ', '_') + '.pkl'
    joblib.dump(model, filename)
    print(f"Saved: {filename}")

with open('feature_names.json', 'w') as f:
    json.dump(feature_names, f)

with open('metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\nAll models saved! Run 'streamlit run app.py' to launch the web app.")